# 01 — M0 RGB Baseline

Trains M0: stock YOLOv8s, no augmentation, no FEM/POPAM, no depth, no visibility head.

**Protocol:**
1. Run smoke test locally first (`run_mode=local`, 3 epochs, 100 samples)
2. If smoke test passes, run full training on Kaggle/cloud (`run_mode=kaggle`)
3. Records AP_easy, AP_mod, AP_hard → `results/tables/val_results_all_models.csv`

**Prerequisite:** notebook `00_data_verification` must have passed.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import RunConfig, RunMode, load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuration ──────────────────────────────────────────────────────────────
# Change RUN_MODE to 'kaggle' or 'colab' for full training
RUN_MODE = 'local'   # 'local' | 'kaggle' | 'colab'
SMOKE_TEST = (RUN_MODE == 'local')

cfg = load_config(ROOT / 'configs' / 'm0_baseline.yaml', overrides={
    'run_mode': RUN_MODE,
    'epochs': 3 if SMOKE_TEST else 100,
    'data_limit': 100 if SMOKE_TEST else None,
})
cfg.ensure_dirs()
set_all_seeds(cfg.seed)

print(f'Model: {cfg.model_id}  |  Mode: {cfg.mode}  |  Epochs: {cfg.epochs}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ── Unit tests (mandatory before any training) ─────────────────────────────────
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / 'train.py'),
     '--config', str(ROOT / 'configs' / 'm0_baseline.yaml'),
     '--run_mode', 'local', '--epochs', '1', '--data_limit', '4'],
    capture_output=True, text=True, cwd=ROOT
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Unit tests failed — do not proceed with training.')
print('\n[OK] Unit tests passed.')

In [ ]:
# ── Build datasets ─────────────────────────────────────────────────────────────
train_ds = KITTIDataset(
    cfg.kitti_root, split='train', imgsz=cfg.imgsz,
    data_limit=cfg.data_limit
)
val_ds = KITTIDataset(
    cfg.kitti_root, split='val', imgsz=cfg.imgsz
)

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch, shuffle=True,
    collate_fn=collate_fn, num_workers=4 if not SMOKE_TEST else 0,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch, shuffle=False,
    collate_fn=collate_fn, num_workers=4 if not SMOKE_TEST else 0
)

print(f'Train samples: {len(train_ds)}  |  Val samples: {len(val_ds)}')

In [ ]:
# ── Load YOLOv8s (M0 — stock, no modifications) ────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'YOLOv8s loaded. Parameters: {sum(p.numel() for p in model.model.parameters()):,}')

In [ ]:
# ── Training with ExperimentLogger ─────────────────────────────────────────────
# Note: Full Ultralytics training uses model.train(). This cell sets up the logger
# and runs training, capturing per-epoch metrics for our custom KITTI AP evaluation.

logger = ExperimentLogger(
    run_id=f'M0_KITTI_p0.0_seed{cfg.seed}',
    log_dir=cfg.log_dir,
    config=vars(cfg),
)

with logger:
    # Ultralytics training (handles optimiser, LR schedule, checkpointing)
    model.train(
        data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),  # Ultralytics dataset config
        epochs=cfg.epochs,
        imgsz=cfg.imgsz,
        batch=cfg.batch,
        optimizer=cfg.optimizer,
        lr0=cfg.lr0,
        lrf=cfg.lrf,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
        warmup_epochs=cfg.warmup_epochs,
        amp=cfg.amp,
        seed=cfg.seed,
        project=str(cfg.checkpoint_dir),
        name=cfg.model_id,
        exist_ok=True,
        device=device,
    )

    # ── Custom KITTI AP evaluation on val set ──────────────────────────────────
    best_ckpt = cfg.checkpoint_dir / cfg.model_id / 'weights' / 'best.pt'
    best_model = YOLO(str(best_ckpt))
    best_model.model.eval()

    predictions, annotations = [], []
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image'].to(device)
            results = best_model(imgs, verbose=False)
            for i, r in enumerate(results):
                img_id = batch['image_id'][i]
                boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                predictions.append({'image_id': img_id, 'boxes': boxes, 'scores': scores})
                annotations.append(sample_to_annotation(batch, i))

    ap_easy = compute_kitti_ap(predictions, annotations, 'easy')
    ap_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
    ap_hard = compute_kitti_ap(predictions, annotations, 'hard')

    print(f'\nM0 Val Results:')
    print(f'  AP_easy:  {ap_easy:.2f}')
    print(f'  AP_mod:   {ap_mod:.2f}')
    print(f'  AP_hard:  {ap_hard:.2f}')

    logger.log_metrics({'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard})
    summary = logger.summary()

In [ ]:
# ── Append results to val_results_all_models.csv ──────────────────────────────
results_csv = TABLES_DIR / 'val_results_all_models.csv'

row = {
    'model_id': 'M0',
    'description': 'RGB Baseline (no aug, no depth, stock YOLOv8s)',
    'AP_easy': ap_easy,
    'AP_mod': ap_mod,
    'AP_hard': ap_hard,
    'aug_p': 0.0,
    'use_fem': False,
    'use_popam': False,
    'fusion_type': 'none',
    'use_vis_head': False,
    'seed': cfg.seed,
    'epochs': cfg.epochs,
}

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df = df[df['model_id'] != 'M0']  # replace if re-running
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])

df.to_csv(results_csv, index=False)
print(f'Results saved to: {results_csv}')
df